# Traffic Sign Inference Pipeline with Depth + Geolocation

This notebook runs the full inference pipeline for a traffic sign image:

1. YOLO detection + coarse classification.
2. Cosine-similarity fine classification within the coarse group.
3. Depth-Anything-V2 metric depth estimation in meters.
4. Pixel-to-camera 3D reconstruction using intrinsics.
5. Camera-to-world projection using an external extrinsic matrix.
6. World-to-GPS conversion using a geodetic anchor.

Important: depth alone is not enough to recover GPS. You need a calibrated camera intrinsic matrix, a world pose / extrinsic matrix, and a reference geodetic origin for the world frame.

The notebook assumes you already generated the cosine-classifier artifacts in `code/cosine_fine_classifier/` using `classifier_cosine_setup.ipynb`.

In [ ]:
import json
import math
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

try:
    from transformers import AutoImageProcessor, AutoModel, AutoModelForDepthEstimation
except ImportError as exc:
    raise ImportError(
        'Please install transformers first: pip install transformers'
    ) from exc

from ultralytics import YOLO

PROJECT_ROOT = Path('/home/minh-le-vo-nhat/Documents/Minh-DUT/Ky-8-2025-2026/BigData/BTNhom')
DATA_ROOT = PROJECT_ROOT / 'Data'
CODE_ROOT = PROJECT_ROOT / 'code'
ARTIFACT_DIR = CODE_ROOT / 'cosine_fine_classifier'

MEAN_NPZ_PATH = ARTIFACT_DIR / 'fine_class_means.npz'
INDEX2FINE_JSON_PATH = ARTIFACT_DIR / 'index_to_fine_class.json'
COARSE2IDXS_JSON_PATH = ARTIFACT_DIR / 'coarse_to_fine_indices.json'
META_JSON_PATH = ARTIFACT_DIR / 'embed_metadata.json'

DEFAULT_YOLO_MODEL_PATH = CODE_ROOT / 'runs' / 'traffic_sign_yolov8_highlevel' / 'weights' / 'best.pt'
DEFAULT_DEPTH_MODEL_ID = 'depth-anything/Depth-Anything-V2-Metric-Outdoor-Base-hf'
DEFAULT_DINO_MODEL_ID = 'facebook/dinov2-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class GeoAnchor:
    lat0: float
    lon0: float
    alt0: float

print(f'Device: {DEVICE}')
print(f'Artifact dir exists: {ARTIFACT_DIR.exists()}')

In [ ]:
def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), eps, None)


def load_artifacts(artifact_dir: Path = ARTIFACT_DIR):
    means = np.load(artifact_dir / 'fine_class_means.npz')['mean_vectors'].astype(np.float32)
    with open(artifact_dir / 'index_to_fine_class.json', 'r', encoding='utf-8') as f:
        index_to_fine_json = json.load(f)
    with open(artifact_dir / 'coarse_to_fine_indices.json', 'r', encoding='utf-8') as f:
        coarse_to_indices_json = json.load(f)

    index_to_fine = {int(k): v for k, v in index_to_fine_json.items()}
    coarse_to_indices = {k: [int(x) for x in v] for k, v in coarse_to_indices_json.items()}
    return means, index_to_fine, coarse_to_indices


def load_dinov2(model_id: str = DEFAULT_DINO_MODEL_ID, device: str = DEVICE):
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(device)
    model.eval()
    return processor, model


def embed_crop_dinov2(crop_bgr: np.ndarray, processor, model, device: str = DEVICE) -> np.ndarray:
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(images=crop_rgb, return_tensors='pt', input_data_format='channels_last')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            embedding = outputs.pooler_output
        else:
            embedding = outputs.last_hidden_state[:, 0, :]
        embedding = F.normalize(embedding, dim=1)
    return embedding.squeeze(0).detach().cpu().numpy().astype(np.float32)


def classify_fine_from_coarse(
    crop_bgr: np.ndarray,
    coarse_label: str,
    processor,
    model,
    mean_vectors: np.ndarray,
    index_to_fine_map: dict[int, str],
    coarse_to_indices_map: dict[str, list[int]],
) -> tuple[str | None, float | None]:
    candidate_indices = coarse_to_indices_map.get(coarse_label, [])
    if not candidate_indices:
        return None, None

    emb = embed_crop_dinov2(crop_bgr, processor, model)
    emb = emb / np.clip(np.linalg.norm(emb), 1e-12, None)
    candidate_matrix = mean_vectors[candidate_indices]
    sims = candidate_matrix @ emb
    best_local_idx = int(np.argmax(sims))
    best_global_idx = int(candidate_indices[best_local_idx])
    return index_to_fine_map[best_global_idx], float(sims[best_local_idx])


def load_depth_model(model_id: str = DEFAULT_DEPTH_MODEL_ID, device: str = DEVICE):
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForDepthEstimation.from_pretrained(model_id).to(device)
    model.eval()
    return processor, model


def estimate_metric_depth_map(image_bgr: np.ndarray, processor, model, device: str = DEVICE) -> np.ndarray:
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(image_rgb)
    inputs = processor(images=pil_image, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth
        depth = F.interpolate(
            predicted_depth.unsqueeze(1),
            size=image_bgr.shape[:2],
            mode='bicubic',
            align_corners=False,
        ).squeeze(1)
    return depth.squeeze(0).detach().cpu().numpy().astype(np.float32)


def sample_depth_at_box(depth_map: np.ndarray, box_xyxy: tuple[int, int, int, int], strategy: str = 'median') -> float:
    x1, y1, x2, y2 = box_xyxy
    h, w = depth_map.shape[:2]
    x1 = max(0, min(int(x1), w - 1))
    y1 = max(0, min(int(y1), h - 1))
    x2 = max(0, min(int(x2), w))
    y2 = max(0, min(int(y2), h))
    if x2 <= x1 or y2 <= y1:
        return float('nan')
    region = depth_map[y1:y2, x1:x2]
    region = region[np.isfinite(region)]
    region = region[region > 0]
    if region.size == 0:
        return float('nan')
    if strategy == 'mean':
        return float(region.mean())
    if strategy == 'center':
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        return float(depth_map[cy, cx])
    return float(np.median(region))


def pixel_to_camera(pixel_xy: tuple[float, float], depth_m: float, K: np.ndarray) -> np.ndarray:
    u, v = pixel_xy
    fx = float(K[0, 0])
    fy = float(K[1, 1])
    cx = float(K[0, 2])
    cy = float(K[1, 2])
    z = float(depth_m)
    x = (u - cx) * z / fx
    y = (v - cy) * z / fy
    return np.array([x, y, z], dtype=np.float64)


def transform_point(T_4x4: np.ndarray, point_xyz: np.ndarray) -> np.ndarray:
    point_h = np.array([point_xyz[0], point_xyz[1], point_xyz[2], 1.0], dtype=np.float64)
    out = T_4x4 @ point_h
    return out[:3] / out[3]


WGS84_A = 6378137.0
WGS84_F = 1.0 / 298.257223563
WGS84_E2 = WGS84_F * (2.0 - WGS84_F)


def geodetic_to_ecef(lat_deg: float, lon_deg: float, alt_m: float) -> np.ndarray:
    lat = math.radians(lat_deg)
    lon = math.radians(lon_deg)
    sin_lat = math.sin(lat)
    cos_lat = math.cos(lat)
    sin_lon = math.sin(lon)
    cos_lon = math.cos(lon)
    n = WGS84_A / math.sqrt(1.0 - WGS84_E2 * sin_lat * sin_lat)
    x = (n + alt_m) * cos_lat * cos_lon
    y = (n + alt_m) * cos_lat * sin_lon
    z = (n * (1.0 - WGS84_E2) + alt_m) * sin_lat
    return np.array([x, y, z], dtype=np.float64)


def ecef_to_geodetic(x: float, y: float, z: float) -> tuple[float, float, float]:
    lon = math.atan2(y, x)
    p = math.sqrt(x * x + y * y)
    lat = math.atan2(z, p * (1.0 - WGS84_E2))
    for _ in range(10):
        sin_lat = math.sin(lat)
        n = WGS84_A / math.sqrt(1.0 - WGS84_E2 * sin_lat * sin_lat)
        alt = p / max(math.cos(lat), 1e-12) - n
        lat_new = math.atan2(z, p * (1.0 - WGS84_E2 * n / (n + alt)))
        if abs(lat_new - lat) < 1e-12:
            lat = lat_new
            break
        lat = lat_new
    sin_lat = math.sin(lat)
    n = WGS84_A / math.sqrt(1.0 - WGS84_E2 * sin_lat * sin_lat)
    alt = p / max(math.cos(lat), 1e-12) - n
    return math.degrees(lat), math.degrees(lon), alt


def enu_rotation_matrix(lat0_deg: float, lon0_deg: float) -> np.ndarray:
    lat0 = math.radians(lat0_deg)
    lon0 = math.radians(lon0_deg)
    sin_lat = math.sin(lat0)
    cos_lat = math.cos(lat0)
    sin_lon = math.sin(lon0)
    cos_lon = math.cos(lon0)
    return np.array(
        [
            [-sin_lon, cos_lon, 0.0],
            [-sin_lat * cos_lon, -sin_lat * sin_lon, cos_lat],
            [cos_lat * cos_lon, cos_lat * sin_lon, sin_lat],
        ],
        dtype=np.float64,
    )


def enu_to_ecef(enu_xyz: np.ndarray, anchor: GeoAnchor) -> np.ndarray:
    ecef_anchor = geodetic_to_ecef(anchor.lat0, anchor.lon0, anchor.alt0)
    rot = enu_rotation_matrix(anchor.lat0, anchor.lon0)
    return ecef_anchor + rot.T @ enu_xyz.astype(np.float64)


def world_to_gps(world_xyz: np.ndarray, anchor: GeoAnchor) -> tuple[float, float, float]:
    ecef = enu_to_ecef(world_xyz, anchor)
    return ecef_to_geodetic(float(ecef[0]), float(ecef[1]), float(ecef[2]))


def build_extrinsic_matrix(R_wc: np.ndarray, t_wc: np.ndarray) -> np.ndarray:
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = R_wc
    T[:3, 3] = t_wc.reshape(3)
    return T

In [ ]:
mean_vectors, index_to_fine_map, coarse_to_indices_map = load_artifacts()
dino_processor, dino_model = load_dinov2()
depth_processor, depth_model = load_depth_model()

with open(META_JSON_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)

print('Loaded embeddings:', mean_vectors.shape)
print('Loaded fine classes:', len(index_to_fine_map))
print('Loaded coarse groups:', len(coarse_to_indices_map))
print('Embedding model:', meta.get('model_id'))
print('Depth model:', DEFAULT_DEPTH_MODEL_ID)

## Approximate calibration from capture metadata

Use this section when you only have GPS location, timestamp, compass heading, device orientation, altitude, speed, and camera information from a smart camera or Mapillary-like capture.

The result is approximate: the intrinsic matrix is estimated from camera metadata, and the extrinsic matrix is constructed from geotagged pose and heading/orientation assumptions. This is useful for rough localization, not for precise surveying.

In [ ]:
def estimate_intrinsic_matrix_from_camera_info(
    image_width: int,
    image_height: int,
    focal_length_mm: float | None = None,
    sensor_width_mm: float | None = None,
    sensor_height_mm: float | None = None,
    focal_length_35mm_equiv: float | None = None,
) -> np.ndarray:
    if focal_length_mm is not None and sensor_width_mm is not None and sensor_height_mm is not None:
        fx = float(image_width) * float(focal_length_mm) / float(sensor_width_mm)
        fy = float(image_height) * float(focal_length_mm) / float(sensor_height_mm)
    elif focal_length_35mm_equiv is not None:
        fx = float(image_width) * float(focal_length_35mm_equiv) / 36.0
        fy = fx
    else:
        fx = 0.9 * float(image_width)
        fy = 0.9 * float(image_height)

    cx = float(image_width) / 2.0
    cy = float(image_height) / 2.0
    return np.array([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]], dtype=np.float64)


def heading_to_rotation_matrix(heading_deg: float, pitch_deg: float = 0.0, roll_deg: float = 0.0) -> np.ndarray:
    # Compass heading is clockwise from north; convert to an ENU yaw angle.
    yaw = math.radians(90.0 - heading_deg)
    pitch = math.radians(pitch_deg)
    roll = math.radians(roll_deg)

    cy = math.cos(yaw)
    sy = math.sin(yaw)
    cp = math.cos(pitch)
    sp = math.sin(pitch)
    cr = math.cos(roll)
    sr = math.sin(roll)

    Rz = np.array([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]], dtype=np.float64)
    Ry = np.array([[cp, 0.0, sp], [0.0, 1.0, 0.0], [-sp, 0.0, cp]], dtype=np.float64)
    Rx = np.array([[1.0, 0.0, 0.0], [0.0, cr, -sr], [0.0, sr, cr]], dtype=np.float64)
    return Rz @ Ry @ Rx


def estimate_extrinsic_matrix_from_capture_metadata(
    latitude: float,
    longitude: float,
    altitude_m: float,
    heading_deg: float,
    timestamp: str | None = None,
    speed_mps: float | None = None,
    pitch_deg: float = 0.0,
    roll_deg: float = 0.0,
    camera_height_above_ground_m: float = 1.5,
    world_frame: str = 'enu',
) -> tuple[np.ndarray, GeoAnchor]:
    _ = timestamp, speed_mps
    anchor = GeoAnchor(lat0=float(latitude), lon0=float(longitude), alt0=float(altitude_m))
    pose_xyz_world = np.array([0.0, 0.0, float(camera_height_above_ground_m)], dtype=np.float64)

    if world_frame.lower() != 'enu':
        raise ValueError(
            "world_frame='enu' is currently supported for approximate geotagged pose reconstruction."
        )

    R_wc = heading_to_rotation_matrix(heading_deg, pitch_deg=pitch_deg, roll_deg=roll_deg)
    T_world_camera = build_extrinsic_matrix(R_wc, pose_xyz_world)
    return T_world_camera, anchor


def build_approximate_calibration_from_capture_metadata(
    image_width: int,
    image_height: int,
    latitude: float,
    longitude: float,
    altitude_m: float,
    heading_deg: float,
    timestamp: str | None = None,
    speed_mps: float | None = None,
    pitch_deg: float = 0.0,
    roll_deg: float = 0.0,
    camera_height_above_ground_m: float = 1.5,
    focal_length_mm: float | None = None,
    sensor_width_mm: float | None = None,
    sensor_height_mm: float | None = None,
    focal_length_35mm_equiv: float | None = None,
) -> tuple[np.ndarray, np.ndarray, GeoAnchor]:
    K = estimate_intrinsic_matrix_from_camera_info(
        image_width=image_width,
        image_height=image_height,
        focal_length_mm=focal_length_mm,
        sensor_width_mm=sensor_width_mm,
        sensor_height_mm=sensor_height_mm,
        focal_length_35mm_equiv=focal_length_35mm_equiv,
    )
    T_world_camera, geo_anchor = estimate_extrinsic_matrix_from_capture_metadata(
        latitude=latitude,
        longitude=longitude,
        altitude_m=altitude_m,
        heading_deg=heading_deg,
        timestamp=timestamp,
        speed_mps=speed_mps,
        pitch_deg=pitch_deg,
        roll_deg=roll_deg,
        camera_height_above_ground_m=camera_height_above_ground_m,
    )
    return K, T_world_camera, geo_anchor


print('Approximate calibration helpers ready.')

## Example using metadata-based approximate calibration

Fill in the values from your smart camera or Mapillary capture. If your camera app stores focal length or 35mm equivalent focal length, use it for a better intrinsic estimate.

In [ ]:
capture_width = 1920
capture_height = 1080
capture_latitude = 21.027763
capture_longitude = 105.834160
capture_altitude_m = 30.0
capture_heading_deg = 90.0
capture_timestamp = '2026-05-26T10:00:00Z'
capture_speed_mps = 0.0
capture_pitch_deg = 0.0
capture_roll_deg = 0.0
capture_camera_height_above_ground_m = 1.5
capture_focal_length_35mm_equiv = 26.0

K_approx, T_world_camera_approx, geo_anchor_approx = build_approximate_calibration_from_capture_metadata(
    image_width=capture_width,
    image_height=capture_height,
    latitude=capture_latitude,
    longitude=capture_longitude,
    altitude_m=capture_altitude_m,
    heading_deg=capture_heading_deg,
    timestamp=capture_timestamp,
    speed_mps=capture_speed_mps,
    pitch_deg=capture_pitch_deg,
    roll_deg=capture_roll_deg,
    camera_height_above_ground_m=capture_camera_height_above_ground_m,
    focal_length_35mm_equiv=capture_focal_length_35mm_equiv,
)

print('Approximate K:\n', K_approx)
print('Approximate T_world_camera:\n', T_world_camera_approx)
print('Geo anchor:', geo_anchor_approx)

## End-to-end inference

The function below performs the full pipeline on one image:

- detect with YOLO,
- map the detection to a coarse label,
- classify the crop to the nearest fine class within that coarse group,
- estimate metric depth,
- project the detection center into camera coordinates,
- convert to world coordinates with a 4x4 extrinsic matrix,
- convert world coordinates to GPS with a geodetic anchor.

If your extrinsic matrix does not represent a local metric world frame aligned to a GPS anchor, do not treat the GPS output as valid.

In [ ]:
def infer_traffic_sign_geolocation(
    image_path: str | Path,
    yolo_model_path: str | Path = DEFAULT_YOLO_MODEL_PATH,
    yolo_coarse_names: list[str] | None = None,
    K: np.ndarray | None = None,
    T_world_camera: np.ndarray | None = None,
    geo_anchor: GeoAnchor | None = None,
    conf: float = 0.25,
    depth_sample_strategy: str = 'median',
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    image_path = Path(image_path)
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')

    if K is None:
        raise ValueError('K is required: pass the 3x3 camera intrinsic matrix.')
    if T_world_camera is None:
        raise ValueError('T_world_camera is required: pass the 4x4 extrinsic matrix.')
    if geo_anchor is None:
        raise ValueError('geo_anchor is required: pass a GeoAnchor with lat0, lon0, alt0.')

    yolo_model = YOLO(str(yolo_model_path))
    result = yolo_model.predict(source=str(image_path), conf=conf, verbose=False)[0]
    image_display = image_bgr.copy()

    if yolo_coarse_names is None:
        if hasattr(result, 'names') and result.names is not None:
            yolo_coarse_names = [result.names[i] for i in range(len(result.names))]
        else:
            raise ValueError('yolo_coarse_names is required when the model does not expose class names.')

    depth_map = estimate_metric_depth_map(image_bgr, depth_processor, depth_model)
    rows = []

    for det_id, box in enumerate(result.boxes):
        coarse_idx = int(box.cls.item())
        coarse_label = yolo_coarse_names[coarse_idx]

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = max(0, x2), max(0, y2)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = image_bgr[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        fine_label, cosine_sim = classify_fine_from_coarse(
            crop_bgr=crop,
            coarse_label=coarse_label,
            processor=dino_processor,
            model=dino_model,
            mean_vectors=mean_vectors,
            index_to_fine_map=index_to_fine_map,
            coarse_to_indices_map=coarse_to_indices_map,
        )

        depth_m = sample_depth_at_box(depth_map, (x1, y1, x2, y2), strategy=depth_sample_strategy)
        if not np.isfinite(depth_m) or depth_m <= 0:
            continue

        center_u = float((x1 + x2) / 2.0)
        center_v = float((y1 + y2) / 2.0)
        camera_xyz = pixel_to_camera((center_u, center_v), depth_m, K)
        world_xyz = transform_point(T_world_camera, camera_xyz)
        lat, lon, alt = world_to_gps(world_xyz, geo_anchor)

        cv2.rectangle(image_display, (x1, y1), (x2, y2), (0, 0, 255), 2)
        label_text = f'{coarse_label} | {fine_label or "unknown"} | z={depth_m:.2f}m'
        cv2.putText(
            image_display,
            label_text,
            (x1, max(0, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

        rows.append(
            {
                'det_id': det_id,
                'coarse_label': coarse_label,
                'fine_label': fine_label,
                'cosine_similarity': cosine_sim,
                'confidence_yolo': float(box.conf.item()),
                'bbox_x1': int(x1),
                'bbox_y1': int(y1),
                'bbox_x2': int(x2),
                'bbox_y2': int(y2),
                'depth_m': float(depth_m),
                'camera_x_m': float(camera_xyz[0]),
                'camera_y_m': float(camera_xyz[1]),
                'camera_z_m': float(camera_xyz[2]),
                'world_x_m': float(world_xyz[0]),
                'world_y_m': float(world_xyz[1]),
                'world_z_m': float(world_xyz[2]),
                'latitude': float(lat),
                'longitude': float(lon),
                'altitude_m': float(alt),
            }
        )

    return pd.DataFrame(rows), depth_map, image_display


def make_example_intrinsics(image_width: int, image_height: int, fx: float, fy: float) -> np.ndarray:
    cx = image_width / 2.0
    cy = image_height / 2.0
    return np.array([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]], dtype=np.float64)

## Example configuration

Replace the placeholder calibration values with your real camera calibration and pose.

- `K` is the 3x3 camera intrinsic matrix.
- `T_world_camera` is the 4x4 transform from camera coordinates to your world frame.
- `GeoAnchor` is the latitude/longitude/altitude of the world-frame origin.

If your world frame is already an ENU frame anchored at a GPS point, the GPS conversion below is valid.

In [ ]:
# Example placeholders. Replace these with your actual calibration values.
image_path = PROJECT_ROOT / 'image.png'

# Example intrinsic matrix; replace with your calibrated K.
K = np.array(
    [
        [1200.0, 0.0, 960.0],
        [0.0, 1200.0, 540.0],
        [0.0, 0.0, 1.0],
    ],
    dtype=np.float64,
)

# Example camera pose in world coordinates. Replace with your real extrinsic matrix.
R_wc = np.eye(3, dtype=np.float64)
t_wc = np.array([0.0, 0.0, 0.0], dtype=np.float64)
T_world_camera = build_extrinsic_matrix(R_wc, t_wc)

# Example GPS anchor for the world-frame origin. Replace with your real anchor point.
geo_anchor = GeoAnchor(lat0=21.027763, lon0=105.834160, alt0=30.0)

# Replace this list with the exact class order of your YOLO model.
# If your trained model already exposes names, the pipeline can infer them automatically.
yolo_coarse_names = ['other-sign', 'complementary', 'information', 'regulatory', 'warning']

df_pred, depth_map, image_display = infer_traffic_sign_geolocation(
    image_path=image_path,
    yolo_model_path=DEFAULT_YOLO_MODEL_PATH,
    yolo_coarse_names=yolo_coarse_names,
    K=K,
    T_world_camera=T_world_camera,
    geo_anchor=geo_anchor,
)

display(df_pred)

import matplotlib.pyplot as plt
plt.figure(figsize=(14, 9))
plt.imshow(cv2.cvtColor(image_display, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

plt.figure(figsize=(12, 6))
plt.imshow(depth_map, cmap='magma')
plt.colorbar(label='metric depth (m)')
plt.title('Depth-Anything V2 metric depth map')
plt.axis('off')
plt.show()

## Notes

1. The depth model provides metric depth, but the absolute accuracy depends on the image domain and the model calibration.
2. GPS recovery requires the world frame to be tied to a geodetic origin. If your `T_world_camera` comes from SLAM, SfM, or visual odometry, you still need an anchor to map the world frame to latitude/longitude/altitude.
3. For more stable localization, sample depth over the whole detection box and ignore tiny or invalid regions.
4. If you want the sign center instead of the bounding-box center, replace the pixel sampling step with the sign keypoint or segmentation centroid.